## Lecture 01 - 04/08/2026

Prof. Pedro Henrique de Almeida Konzen.
Avaliação: artigo sobre um problema a ser resolvido com métodos numéricos vistos na disciplina. Apresentar em formato de seminário.
(Escolher o  tema [linkado com a tese] e um tema da disciplina que possa ser relacionado. O professor irá avaliar se é viável ou não.)

In [4]:
# Exercise 1.2.2:

name = input("Digite seu nome: ")
print(f"Hello, {name}!!!")

Hello, Éderson Passos!!!


## Lecture 04 - 13/08/2026

## 1.1 Band Matrix

In [13]:
size_float = 8
# matrix n x n
n = 100_000
print(f"Storage: {n*n*size_float/1024**3} GB")

Storage: 74.50580596923828 GB


In [14]:
# Example 1.1.2: 1D Poisson Equation
import numpy as np
n = 11
h = 1/(n-1)
xx = np.linspace(0, 1, n)
# Matrix
Ab = np.zeros((3,n))
u_diagonal = 1
l_diagonal = 1

def index(i, j):
    return u_diagonal + i - j, j

function = lambda x: np.pi**2*np.sin(np.pi*x)
b = np.zeros(n)

# Build the matrix
Ab[index(0,0)] = 1.0

for i in range(1, n-1):
    Ab[index(i,i)] = 2.0
    Ab[index(i,i-1)] = -1.0
    Ab[index(i,i+1)] = -1.0
    b[i] = function(xx[i])

Ab[index(n-1,n-1)] = 1.0
print(Ab)
print(b)

[[ 0.  0. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [ 1.  2.  2.  2.  2.  2.  2.  2.  2.  2.  1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1.  0.  0.]]
[0.         3.04987549 5.80120791 7.98467769 9.38655158 9.8696044
 9.38655158 7.98467769 5.80120791 3.04987549 0.        ]


In [15]:
def solve_tribanded(ab, b):
    a = ab.copy()
    x = b.copy()
    n = b.size
    # eliminação
    for i in range(1,n):
        w = a[2,i-1]/a[1,i-1]
        a[1,i] -= w * a[0,i]
        x[i] -= w * x[i-1]
    # resolve
    x[n-1] = x[n-1]/a[1,n-1]
    for i in range(n-2,-1,-1):
        x[i] = (x[i] - a[0,i+1]*x[i+1])/a[1,i]
    return x

u_diagonal = solve_tribanded(Ab, b)
print(u_diagonal)

[  0.          31.15711487  59.26435425  81.57038572  95.8917395
 100.8265417   95.8917395   81.57038572  59.26435425  31.15711487
   0.        ]


## Lecture 18/08/2026

### Exemple 1.1.3.(Poisson 2D Equation)

In [1]:
import numpy as np
from scipy.linalg import solve_banded


# malha
n = 11
h = 1/(n-1)

xx = np.linspace(0, 1, n)
yy = np.linspace(0, 1, n)

# fonte
def f(x,y):
    return 2 * np.pi**2 * np.sin(np.pi*x) * np.sin(np.pi*y)

# sistema discreto
upper = lower = n-2
ab = np.zeros((upper+lower+1, (n-2)**2))
b = np.empty((n-2)**2)

# índice no formato banda
def ind(k, l):
    return upper + k - l, l

for j in np.arange(2,n):
    for i in np.arange(2,n):

        # enumeração dos nodos computacionais
        k = i-2 + (j-2)*(n-2)

        # vetor b
        b[k] = h**2 * f(xx[i-1], yy[j-1])

        # matriz ab
        ab[ind(k,k)] = 4.

        if (j == 2):
            if (i == 2):
                ab[ind(k,k+1)] = -1.
                ab[ind(k,k+n-2)] = -1.
            elif (i <= n-2):
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k+1)] = -1.
                ab[ind(k,k+n-2)] = -1.
            else: # i == n-1
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k+n-2)] = -1.
        elif (j <= n-2):
            if (i == 2):
                ab[ind(k,k-(n-2))] = -1.
                ab[ind(k,k+1)] = -1.
                ab[ind(k,k+n-2)] = -1.
            elif (i <= n-2):
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k-(n-2))] = -1.
                ab[ind(k,k+1)] = -1.
                ab[ind(k,k+n-2)] = -1.
            else: # i == n-1
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k-(n-2))] = -1.
                ab[ind(k,k+n-2)] = -1.
        else: # j == n-1
            if (i == 2):
                ab[ind(k,k-(n-2))] = -1.
                ab[ind(k,k+1)] = -1.
            elif (i <= n-2):
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k-(n-2))] = -1.
                ab[ind(k,k+1)] = -1.
            else: # i == n-1
                ab[ind(k,k-1)] = -1.
                ab[ind(k,k-(n-2))] = -1.

# resolve
u = solve_banded((lower, upper), ab, b)

In [2]:
print(u)

[0.09628078 0.18313693 0.25206635 0.29632177 0.31157115 0.29632177
 0.25206635 0.18313693 0.09628078 0.18313693 0.34834713 0.4794587
 0.5636375  0.59264354 0.5636375  0.4794587  0.34834713 0.18313693
 0.25206635 0.4794587  0.65991828 0.77578047 0.81570386 0.77578047
 0.65991828 0.4794587  0.25206635 0.29632177 0.5636375  0.77578047
 0.91198464 0.95891739 0.91198464 0.77578047 0.5636375  0.29632177
 0.31157115 0.59264354 0.81570386 0.95891739 1.00826542 0.95891739
 0.81570386 0.59264354 0.31157115 0.29632177 0.5636375  0.77578047
 0.91198464 0.95891739 0.91198464 0.77578047 0.5636375  0.29632177
 0.25206635 0.4794587  0.65991828 0.77578047 0.81570386 0.77578047
 0.65991828 0.4794587  0.25206635 0.18313693 0.34834713 0.4794587
 0.5636375  0.59264354 0.5636375  0.4794587  0.34834713 0.18313693
 0.09628078 0.18313693 0.25206635 0.29632177 0.31157115 0.29632177
 0.25206635 0.18313693 0.09628078]
